In [0]:
from pyspark.sql import functions as F

def drop_duplicates(df):
  df = df.dropDuplicates()
  return df

def query():
    query="""
    with situ as (
        select 
            *,
            case when anos_atividade < 3 then "Empresa com até 3 anos"
                when anos_atividade >= 3 and anos_atividade < 20 then "Empresa com 3 a 20 anos"
                when anos_atividade >= 20 and anos_atividade < 30 then "Empresa com 20 a 30 anos"
                when anos_atividade >= 30 and anos_atividade < 40 then "Empresa com 30 a 40 anos"
                when anos_atividade >= 40 and anos_atividade < 50 then "Empresa com 40 a 50 anos"
                when anos_atividade >= 50 then "Empresa com até 50"
            end as tempo_estab
        from temp_view_cnpj_socios_status_cadastral
    )
    select  
        descricao_pais as pais_estab,
        desc_status_cadastral as status_cad_estab,
        descricao_motivo_status_cadastral as motivo_status_estab,
        tempo_estab as tempo_atividade,
        count(cnpj_basico) as quantidade
    from situ
    where cnae_fiscal_principal in (
            "6201500",
            "6201501",
            "6201502",
            "6202300",
            "6203100",
            "6204000",
            "6209100",
            "6311900",
            "6319400"
        )
    group by  
        descricao_pais,
        desc_status_cadastral,
        descricao_motivo_status_cadastral,
        tempo_estab

    """
    return query

def tables_joins():
    mapeamento={
        "cnpj_socios_status_cadastral":"workspace.silver.cnpj_socios_status_cadastral"
    }
    for table in mapeamento:
        df = spark.table(mapeamento[table])
        df = drop_duplicates(df)
        df.createOrReplaceTempView(f"temp_view_{table}")
        print(f"Temp view {table} criada")
    print("Aplicando a query nas tables")
    query_sql = query()
    df_final = spark.sql(query_sql)
    df_final.display()
    return df_final

def write_table(df):
    df.write.format("delta").mode("overwrite").saveAsTable("workspace.gold.tempo_status_societario_ti")
    print("Dados gravados com sucesso")
    return


def main ():
    df_final=tables_joins()
    write_table(df_final)
main()
